# Revisiting SASRec on Amazon Reviews 2023

**Team:** Alina Chuang, Frank Bartkus, Kyle Nguyen, Matthew Mikhin

We re-run the experiments of SASRec (Kang & McAuley, ICDM 2018) [1] on the 2023 release of the
Amazon review data [2]. Our main question is whether the paper's sampled evaluation and a full
ranking over all items put the methods in the same order.

This notebook is the whole project and runs from top to bottom.

**Contents**

0. About: how to run, division of work, contribution statement, use of AI tools
1. Problem definition
2. Settings
3. Data
4. Evaluation
5. Methods
6. Training and tuning
7. Results
8. Sampled evaluation vs full ranking: our main question
9. Demo
10. Discussion: observations, limitations, future work
11. References

## 0. About this notebook

### How to run it

```bash
uv sync                  # once: installs the Python packages from uv.lock into .venv
uv run jupyter lab       # or open the notebook in VS Code and pick the .venv kernel
```

Then run all cells from the top. The first run downloads the data (about 14 MB, Section 3.1). With `RUN_EXPERIMENTS = False` (Section 2) the trained
results are read from `results/` and the notebook runs in minutes; with `True` every model is
trained again, which takes hours.

### Division of work

**Alina Chuang**
- drafted the shared rules every section follows (evaluation, model interface, tuning, result
  files) and chose the dataset category
- shared settings and `set_seed` (Section 2)
- SASRec model and its explanation (Section 5.3)
- the shared training loop `train_model` with early stopping, and the grid search and test
  helpers every trained model uses (Section 6)
- tuned and tested SASRec with 3 seeds, saved the model for the demo (Section 6.1)
- PopRec test results (Section 6.2)

**Frank Bartkus**
- data loading and the leave-one-out split (Section 3.1)
- the metric code (`ranks_to_result`) and the two sanity-check models of the evaluation
  (Section 4)
- first versions of the negatives, the evaluation functions and PopRec (Sections 4, 5.1)

**Kyle Nguyen**
- problem definition and related work (Section 1)
- rewrote the negative sampling, `evaluate_sampled`, `evaluate_full` and `recommend` into
  their final versions (Section 4)
- rewrote PopRec into the shared model interface (Section 5.1)
- BPR model, tuned and tested with 3 seeds (Section 5.2)
- main results table and RQ2 (Section 7)
- RQ1: sampled evaluation vs full ranking (Section 8)
- references (Section 11)

**Matthew Mikhin**
- data checks and exploratory analysis (Section 3.2)
- explanation of the shared model interface (Section 5)
- figures (Sections 3.2, 7, 8) and results by history length (Section 8)
- demo and demo video (Section 9)
- discussion: observations, limitations, future work and the conclusion (Section 10)

### Contribution statement

| Member | Contribution |
|---|---|
| Alina Chuang | |
| Frank Bartkus | |
| Kyle Nguyen | |
| Matthew Mikhin | |

### Use of generative AI tools

| Member | Tool | Used for |
|---|---|---|
| Alina Chuang | | |
| Frank Bartkus | | |
| Kyle Nguyen | | |
| Matthew Mikhin | | |

## 1. Problem definition (Kyle)

A sequential recommender uses a user's past interactions to predict their next item. For
example, someone who reviewed a tennis racket, tennis balls and an overgrip might review a
racket bag next. We want the recommender to put that next product near the top of its list.
We treat every review as an interaction and ignore the star rating.

SASRec [1] uses self-attention to learn which items in the user's history matter for the next
prediction. It uses the order of the items and cannot look at future interactions.

### Why repeat the experiment?

The original paper used older Amazon data. We use Amazon Reviews 2023 [2] to see whether its
findings still hold on newer data. We chose `Industrial_and_Scientific` because it is small
enough to train the models several times on our machines.

### Research questions

1. **RQ1 (main question):** Do sampled evaluation and full ranking put the methods in the
   same order?
2. **RQ2:** Does SASRec still perform better than PopRec and BPR?

Sampled evaluation ranks the correct item against 100 random negatives. Full ranking
compares it against all eligible items.

### Comparison with the original paper

For both Hit@10 and NDCG@10, Table III of the original paper [1] reports:

- **Beauty:** SASRec > PopRec > BPR
- **Games:** SASRec > BPR > PopRec

We compare these orders with our results. We do not expect the exact scores to match because
our dataset is different.

### Related work

BPR [3] learns personal preferences but ignores item order. Krichene and Rendle [4] show that
sampled evaluation can change the order of the methods, which motivates RQ1. Ferrari Dacrema
et al. [5] show why simple baselines need proper tuning, so we also tune BPR on the validation
set.

Our tools are PyTorch [6] for the models, pandas [7] for data and tables, and Matplotlib [8]
for plots.

## 2. Settings (Alina)

SASRec is compared with PopRec and BPR. That comparison only means something if every method
gets the same data, the same test cases, the same negatives, the same metric and the same
tuning. A difference in any of these would show up as a difference between methods, so every
section follows the same rules.

This section holds only the fixed numbers that several sections use, and `set_seed`. They come
first because the code of every later section uses them.

In [1]:
import copy
import os
import random
import time
import urllib.request

import pandas as pd
import torch

# The Amazon Reviews 2023 category we use; why this one: Section 3.2.
CATEGORY = "Industrial_and_Scientific"

# True: train every model again (hours). False: read the saved results from results/.
RUN_EXPERIMENTS = False

# Evaluation, as in the paper (Section IV-D)
EVAL_SEED = 0  # one fixed set of 100 negatives, the same for every method
N_NEGATIVES = 100  # each answer is ranked against 100 items the user never touched
K = 10  # we report Hit@10 and NDCG@10

# Training
TRAIN_SEEDS = [1, 2, 3]  # every trained model runs 3 times; we report mean and std
PATIENCE = 10  # stop after 10 epochs without a better validation NDCG@10 (paper: 20)
MAX_EPOCHS = 100  # never train longer than this, so every run ends
MAX_LEN = 50  # SASRec reads at most the last 50 items (paper IV-C)

A model uses random numbers for its starting weights, for picking negative items and for the
order of the batches. If two runs with the same seed gave different results, our mean over 3
seeds would mean nothing. `set_seed` puts every random number generator we use back to the
same starting point. The training loop calls it before it creates a model.

In [2]:
def set_seed(seed):
    """Fix every random number generator we use, so a run can be repeated exactly."""
    random.seed(seed)  # Python's own generator, for example random.sample
    torch.manual_seed(seed)  # torch on the CPU, and on the GPU too (CUDA or Apple MPS)


# Check: the same seed gives the same numbers.
set_seed(1)
first = torch.rand(3)
set_seed(1)
second = torch.rand(3)
print(first)
print(second)
print("same:", torch.equal(first, second))

tensor([0.7576, 0.2793, 0.4031])
tensor([0.7576, 0.2793, 0.4031])
same: True


## 3. Data

### 3.1 Source and loading (Frank)

**Source.** We use the official 5-core files of Amazon Reviews 2023 [2] for `CATEGORY`, which
the dataset's authors already split with the leave-one-out strategy (`last_out_w_his`): for
every user, the last item is the test item, the second-to-last item is the validation item,
and everything before is for training. The paper uses the same split (Section IV-A). The first
run downloads the three files (train, valid, test; about 14 MB) from the
[official site](https://amazon-reviews-2023.github.io/data_processing/5core.html) into
`data/<CATEGORY>/`; later runs read them from there. Every review counts as one interaction and
the star rating is not used, because the paper treats reviews as implicit feedback.

**How we build each user's sequence, and why:**

| Step | What we do | Why |
|---|---|---|
| Order | the user's train rows sorted by `timestamp` with a **stable** sort, then the valid item, then the test item | rows with the same timestamp keep their file order, so every computer gets the same sequence |
| User numbers | 0, 1, 2, … in order of first appearance in the train file | the same numbers on every computer; a Python `set` would give a different order on every run |
| Item numbers | 0, 1, 2, … in order of first appearance, reading train, then valid, then test | same reason; an item seen only in valid or test still gets a number |
| Repeated user–item pairs | kept as they are in the files (counted in Section 3.2) | the official files are the protocol; changing them would make our data differ from everyone else's |
| Validation | answer = second-to-last item; input = all items before it | leave-one-out (IV-A) |
| Test | answer = last item; input = all items before it, **including the validation item** | the paper does the same (IV-A) |
| The official `history` column | not read | we build every history from our own sequences, so all sections use exactly the same ones |
| Check | the notebook stops if a user does not have exactly one valid row and one test row | the split above needs exactly one of each |

**The names every later section uses:**

| Name | What it is |
|---|---|
| `sequences` | `sequences[u]`: item numbers of user `u`, oldest first: train items, valid item, test item |
| `user_ids`, `item_ids` | the original ids: `user_ids[u]` is the `user_id` of user `u`, `item_ids[j]` the `parent_asin` of item `j` |
| `n_users`, `n_items` | `len(user_ids)`, `len(item_ids)` |
| `train_seqs[u]` | `sequences[u][:-2]`: what models learn from, and the validation input |
| `valid_answers[u]` | `sequences[u][-2]` |
| `test_histories[u]` | `sequences[u][:-1]` |
| `test_answers[u]` | `sequences[u][-1]` |

Item number `j` is column `j` of every score matrix.

In [3]:
# where the official leave-one-out files of every category are
DATA_URL = (
    "https://mcauleylab.ucsd.edu/public_datasets/data/amazon_2023/benchmark/"
    "5core/last_out_w_his/"
)


# reads one official file, and downloads it into data/<category>/ the first time
def read_split(category, split):
    name = category + "." + split + ".csv.gz"
    path = os.path.join("data", category, name)
    if not os.path.exists(path):
        os.makedirs(os.path.dirname(path), exist_ok=True)
        urllib.request.urlretrieve(DATA_URL + name, path)
    # only the three columns we use: no star rating, no official history
    return pd.read_csv(path, usecols=["user_id", "parent_asin", "timestamp"])


# stops the notebook unless every user has exactly one valid row and one test row
def check_one_row_per_user(train_df, valid_df, test_df):
    train_users = set(train_df["user_id"])
    for name, df in [("valid", valid_df), ("test", test_df)]:
        # how many rows every user has in this file
        counts = df["user_id"].value_counts()
        missing = len(train_users - set(counts.index))
        if missing or (counts != 1).any():
            raise ValueError(
                f"every user needs exactly one {name} row: {missing} users have none, "
                f"{(counts > 1).sum()} users have more than one"
            )


# returns non duplicate values in list in order of earliest occurance
def unique(m_list):
    m_list = list(m_list)
    # pairs every value with its position, then keeps the first position of each value
    positions = pd.DataFrame(zip(m_list, range(len(m_list)), strict=True))
    return positions.groupby(0).min().sort_values(by=1).index.to_list()


def load_data(category):
    # the train, valid and test sets from the data files
    df_list = [read_split(category, t) for t in ["train", "valid", "test"]]
    train_df, valid_df, test_df = df_list
    check_one_row_per_user(train_df, valid_df, test_df)
    # the unique user and item ids sorted by earliest occurrence
    user_ids = unique(train_df["user_id"])
    item_ids = unique([d for df in df_list for d in df["parent_asin"]])
    # series to convert item ids to its respective indices
    item_id_to_index = pd.Series(range(len(item_ids)), index=item_ids)
    # the user and item ids sorted by timestamp for each file in the datafiles
    sorted_dfs = [
        df.sort_values(by="timestamp", kind="stable").drop(columns="timestamp")
        for df in df_list
    ]
    # groups the item ids by the user ids
    sorted_dfs = [df.groupby("user_id").agg(list)["parent_asin"] for df in sorted_dfs]
    # converts the item ids into their indices
    sorted_dfs = [
        pd.Series(
            [[item_id_to_index[i] for i in items] for items in df], index=df.index
        )
        for df in sorted_dfs
    ]
    # the train items, the valid item and the test item of every user, in one list
    sequences = [[int(d) for df in sorted_dfs for d in df[u]] for u in user_ids]
    # returns the sequences, the user ids, and the item ids
    return sequences, user_ids, item_ids

In [4]:
sequences, user_ids, item_ids = load_data(CATEGORY)
n_users, n_items = len(user_ids), len(item_ids)
# should match the official table: 51.0K users, 25.8K items, 412.9K interactions
n_interactions = sum(len(s) for s in sequences)
print(f"users: {n_users}  items: {n_items}  interactions: {n_interactions}")

users: 50985  items: 25848  interactions: 412947


In [5]:
# splits every user's sequence into the four lists the later sections use
def leave_one_out(sequences):
    for s in sequences:
        # every user needs at least one train item, the valid item and the test item
        if len(s) < 3:
            raise ValueError("every sequence must have at least 3 items")
    # train items: everything except the last two
    train_seqs = [s[:-2] for s in sequences]
    # the valid answer: the second-to-last item
    valid_answers = [s[-2] for s in sequences]
    # the test input: everything before the last item, so it includes the valid item
    test_histories = [s[:-1] for s in sequences]
    # the test answer: the last item
    test_answers = [s[-1] for s in sequences]
    return train_seqs, valid_answers, test_histories, test_answers

In [6]:
train_seqs, valid_answers, test_histories, test_answers = leave_one_out(sequences)

### 3.2 Data checks and exploratory analysis (Matthew)

> **TODO (Matthew)** Write this section as report text and code, then delete this box.
> - [ ] Why `Industrial_and_Scientific`, in your own words (you present the data). The facts:
>
>   | | Paper: Amazon Beauty (Table II) | Paper: Amazon Games (Table II) | Ours: `Industrial_and_Scientific` 2023, 5-core |
>   |---|---|---|---|
>   | users | 52,024 | 31,013 | 50,985 |
>   | items | 57,289 | 23,715 | 25,848 |
>   | interactions | 0.4M | 0.3M | 412,947 |
>
>   - Small, as the teacher advised, and about the size of the paper's datasets, so every
>     model can be trained several times on a laptop.
>   - The three `last_out_w_his` files are only 7.1 MB, 3.5 MB and 3.7 MB.
>   - Different products from the paper's Beauty and Games. That's fine, because we compare
>     the order of the methods, not the exact numbers (teacher's feedback).
>   - Not `Video_Games` (94.8K users, 814.6K interactions, about twice the work), not
>     `Beauty_and_Personal_Care` (729.6K users), not `All_Beauty` (only 253 users after
>     5-core).
> - [ ] A size table like the paper's Table II, computed from `sequences`: users, items,
>   interactions, average interactions per user and per item, next to the paper's numbers.
>   The teacher asked us to report the size after filtering.
> - [ ] One figure: number of interactions per user, log scale on both axes.
> - [ ] The two checks the proposal promised: how many repeated user–item pairs, and how many
>   timestamp ties inside one user's history (read the timestamps with `read_split`).

## 4. Evaluation (Frank)

Every method is measured in exactly the same way:

| Decision | Rule | Why |
|---|---|---|
| Negatives | for every user, separately for validation and for test: 100 items drawn uniformly, without repeats, from the items that appear **nowhere** in the user's sequence | the paper's sampled evaluation (IV-D); an item the user did interact with is not a fair negative |
| Random generator for negatives | `random.Random(EVAL_SEED)`, users in index order, all validation negatives first, then all test negatives | the same negatives on every computer and for every method |
| Candidate format | a tensor of shape `(n_users, 101)`: row `u` is user `u`, **column 0 is the answer**, columns 1 to 100 the negatives | the metric code finds the answer without a search |
| Sampled rank `r` | 1 + the number of negatives whose score is **≥** the answer's score | ties count against the answer, so a model that gives every item the same score gets rank 101, not 1 |
| Full-ranking rank `r` | 1 + the number of other items, not in the input history, whose score is **≥** the answer's score | the same tie rule over all items; items the user already has are left out, except the answer itself |
| Hit@10 | 1 if `r ≤ 10`, else 0; averaged over users | paper IV-D |
| NDCG@10 | `1 / log2(r + 1)` if `r ≤ 10`, else 0; averaged over users | paper IV-D; with one answer this is the whole formula |

`evaluate_sampled(model, histories, candidates)` and `evaluate_full(model, histories, answers)`
return `{"hit@10": ..., "ndcg@10": ..., "ranks": ...}`, and `recommend(model, users, histories, k)`
gives the top `k` items per user with the input history removed (used by the demo).

In [7]:
# set_seed(EVAL_SEED) reset both Python and PyTorch’s global randomness, so every run of
# the notebook uses the same 100 negatives for every user.
# keep negatives same for every run
eval_rng = random.Random(EVAL_SEED)
item_set = set(range(n_items))

In [8]:
# Exclude all known interactions from negatives
valid_candidates = []
for sequence, answer in zip(
    sequences, valid_answers, strict=True
):  # pairs each answer with its user’s full sequence
    unseen_items = sorted(item_set - set(sequence))
    negatives = eval_rng.sample(unseen_items, N_NEGATIVES)
    valid_candidates.append([answer] + negatives)

test_candidates = []
for sequence, answer in zip(sequences, test_answers, strict=True):
    unseen_items = sorted(item_set - set(sequence))
    negatives = eval_rng.sample(unseen_items, N_NEGATIVES)
    test_candidates.append([answer] + negatives)

valid_candidates = torch.tensor(valid_candidates, dtype=torch.long)
test_candidates = torch.tensor(test_candidates, dtype=torch.long)

In [9]:
class ConstantModel:  # gives every item the same score
    def __init__(self, n_items):
        self.n_items = n_items

    def score(self, users, histories):
        return torch.zeros(len(users), self.n_items)


class AnswerModel:  # FOR TESTING ONLY
    def __init__(self, n_items, answers):
        self.n_items = n_items
        self.answers = answers

    def score(self, users_, histories):
        s = torch.zeros(len(users_), self.n_items)
        rows = torch.arange(len(users_))
        s[rows, torch.tensor([self.answers[u] for u in users_])] = 1.0
        return s

In [10]:
def ranks_to_result(ranks):
    hit = torch.where(ranks <= K, 1, 0.0).mean()
    ndcg = torch.where(ranks <= K, 1 / torch.log2(ranks + 1), 0).mean()
    return {"hit@" + str(K): hit, "ndcg@" + str(K): ndcg, "ranks": ranks}

In [11]:
# avoid full matrix; no need for gradients here
@torch.no_grad()
def evaluate_sampled(model, histories, candidates, batch_size=256):
    candidates = torch.as_tensor(candidates, dtype=torch.long)
    ranks = []
    for start in range(0, len(histories), batch_size):  # batch scoring
        end = min(start + batch_size, len(histories))
        score = model.score(range(start, end), histories[start:end])
        selected = score.gather(1, candidates[start:end])  # use tensor operations
        ranks.append(1 + (selected[:, 1:] >= selected[:, :1]).sum(dim=1))
    return ranks_to_result(torch.cat(ranks))

In [12]:
# Full ranking takes one answer per user; candidate lists belong to sampled ranking.
@torch.no_grad()
def evaluate_full(model, histories, answers, batch_size=256):
    answers = torch.as_tensor(answers, dtype=torch.long)
    ranks = []
    for start in range(0, len(histories), batch_size):
        end = min(start + batch_size, len(histories))
        score = model.score(
            range(start, end), histories[start:end]
        ).clone()  # use clone to copy before masking
        rows = torch.arange(end - start)
        answer_list = score[rows, answers[start:end]]
        for row, history in enumerate(histories[start:end]):
            score[row, history] = -torch.inf  # exlude seen itgems
        score[rows, answers[start:end]] = -torch.inf
        ranks.append(1 + (score >= answer_list[:, None]).sum(dim=1))
    return ranks_to_result(torch.cat(ranks))

In [13]:
# Batch recommendations to limit memory; inference needs no gradients.
@torch.no_grad()
def recommend(model, users, histories, k, batch_size=256):
    recommendations = []
    for start in range(0, len(users), batch_size):
        end = min(start + batch_size, len(users))
        score = model.score(
            users[start:end], histories[start:end]
        ).clone()  # again, copy b4 masking
        for row, history in enumerate(histories[start:end]):
            score[row, history] = -torch.inf  # exclude seen items
        recommendations.append(torch.topk(score, k, dim=1).indices)
    return torch.cat(recommendations)

In [14]:
const_model_ = ConstantModel(n_items)
ans_model_ = AnswerModel(n_items, valid_answers)

In [15]:
# 0, 0, all 101s
evaluate_sampled(const_model_, train_seqs, valid_candidates)

{'hit@10': tensor(0.),
 'ndcg@10': tensor(0.),
 'ranks': tensor([101, 101, 101,  ..., 101, 101, 101])}

In [16]:
# 0, 0, n_items - len(history) (25844, 25843, 25791,  ..., 25845, 25843, 25845)
evaluate_full(const_model_, train_seqs, valid_answers)

{'hit@10': tensor(0.),
 'ndcg@10': tensor(0.),
 'ranks': tensor([25844, 25843, 25791,  ..., 25845, 25843, 25845])}

In [17]:
recommend(const_model_, range(n_users), train_seqs, K)

tensor([[ 8,  9,  4,  ..., 13, 10,  6],
        [ 3,  9,  1,  ..., 14, 10,  2],
        [ 8,  4,  1,  ..., 66,  2,  6],
        ...,
        [ 8,  9,  4,  ...,  0,  2,  6],
        [ 8,  9,  4,  ...,  0,  2,  6],
        [ 8,  9,  4,  ...,  0,  2,  6]])

In [18]:
# 1, 1, all 1s
evaluate_sampled(ans_model_, train_seqs, valid_candidates)

{'hit@10': tensor(1.),
 'ndcg@10': tensor(1.),
 'ranks': tensor([1, 1, 1,  ..., 1, 1, 1])}

In [19]:
# 1, 1, all 1s
evaluate_full(ans_model_, train_seqs, valid_answers)

{'hit@10': tensor(1.),
 'ndcg@10': tensor(1.),
 'ranks': tensor([1, 1, 1,  ..., 1, 1, 1])}

In [20]:
recommend(ans_model_, range(n_users), train_seqs, K)

tensor([[ 4843,     4,     8,  ...,     5,    13,    10],
        [ 3506,     1,     3,  ...,     0,    14,    10],
        [ 2624,     1,     8,  ...,     5,    66,     2],
        ...,
        [ 9233,     4,     8,  ...,     5,     0,     2],
        [12300,     4,     8,  ...,     5,     0,     2],
        [21668,     4,     8,  ...,     5,     0,     2]])

## 5. Methods

Every method (PopRec, BPR, SASRec) is a class with three methods, so one training loop and one
evaluation work for all of them:

| Method | What it does |
|---|---|
| `__init__(self, n_users, n_items, ...)` | creates the model; settings such as `d` come after the first two arguments, with the paper's value as the default |
| `train_one_epoch(self, train_seqs)` | learns from the training sequences once and returns the average loss as a `float` |
| `score(self, users, histories)` | returns a tensor of shape `(len(users), n_items)`: a score for every item for every case; higher means recommend earlier |

Case `r` is user `users[r]` with input history `histories[r]`. BPR needs the user, SASRec needs
the history, PopRec needs neither.

All models follow these rules:

1. A model learns only from what `train_one_epoch` receives, never from a global variable such
   as `test_answers` or `sequences`.
2. A model never calls `set_seed` or makes its own seeded random generator; the training loop
   seeds everything before the model is created.
3. Every setting is an argument of `__init__`; no numbers hidden inside methods.
4. `score` returns exactly `n_items` columns. A padding item (SASRec) is removed before
   returning.
5. No metric code inside a model; Section 4 measures every model.
6. Everything runs on the CPU.

> **TODO (Matthew)** You present how the parts of the notebook fit together, so rewrite the
> text above in your own words (keep the table), then delete this box.

### 5.1 PopRec (Frank)

PopRec scores every item by how often it appears in `train_seqs`, so every user gets the same
scores.

In [21]:
# refactored the interface to match the other models, so evaluation and the demo can use
# it directly
class PopRec:
    def __init__(self, n_users, n_items):
        self.n_users = n_users
        self.n_items = n_items
        self.counts = torch.zeros(n_items)

    def train_one_epoch(self, train_seqs):
        items = torch.tensor(
            [item for seq in train_seqs for item in seq], dtype=torch.long
        )
        # Recount each time so repeated epochs do not inflate popularity.
        # minlength keeps zero scores for items absent from training.
        self.counts = torch.bincount(items, minlength=self.n_items).float()
        return 0.0

    # Every user gets the same popularity scores without copying the counts.
    def score(self, users, histories):
        return self.counts.expand(len(users), -1)


pop_rec = PopRec(n_users, n_items)
pop_rec.train_one_epoch(train_seqs)

0.0

### 5.2 BPR (Kyle)

BPR [3] learns a vector for each user and item. Their dot product, `P[u] · Q[i]` (`user_emb`
and `item_emb` in the code), is the item's score for that user. We include it as a
personalized baseline: unlike PopRec it can learn different tastes, but it ignores the order
of interactions, unlike SASRec.

Each epoch shuffles the training interactions and pairs each positive item with one uniformly
sampled item absent from that user's **training** history. Validation and test interactions
are not consulted. We minimize `-logsigmoid(positive_score - negative_score)` plus `l2` times
the squared norms of the three sampled vectors, averaged over the batch. `logsigmoid` avoids
numerical overflow when the score difference is large.

We choose Adam with learning rate 0.001 for adaptive updates, batches of 128 to limit memory,
and small normal initial weights (mean 0, standard deviation 0.01) to start with small scores
and distinct vectors. These are our implementation choices; the original BPR paper uses
bootstrap sampling and stochastic gradient descent. Our epoch visits each training interaction
once, including repeats kept by the data loader. Settings are constructor arguments so they
can be inspected and changed. The model uses the shared `train_one_epoch` and `score` interface.

The best grid point was d=50, l2=0.01 with validation NDCG@10 0.349. It reached the 100 epoch limit while still improving slightly, so BPR might gain a little with longer training. With l2=0.001, both sizes overfit after 6 to 12 epochs, and d=50 only clearly beat d=10 with the stronger regularization l2=0.01.



> **TODO (Kyle)** Delete this box when done.
> - [ ] Tune and test BPR with Section 6's `tune_and_test`. Add a code cell right after the
>   `SASREC_GRID` cell in Section 6.1 with:
>   ```python
>   BPR_GRID = [
>       {"d": 10, "l2": 0.001},
>       {"d": 10, "l2": 0.01},
>       {"d": 50, "l2": 0.001},
>       {"d": 50, "l2": 0.01},
>   ]
>   if RUN_EXPERIMENTS:
>       tune_and_test("bpr", BPR, BPR_GRID)
>   show_results("bpr")
>   ```
>   Run it once with `RUN_EXPERIMENTS = True` (about 2 s per epoch on a laptop CPU, so at most
>   about 20 minutes for 4 grid points and 2 extra seeds), set it back to `False`, and commit the
>   updated `results/tuning.csv`, `results/test.csv` and `results/ranks.csv.gz`.
> - [ ] Add one or two sentences on which grid point won and its validation NDCG@10.
> - [ ] If BPR loses to PopRec, first check in `results/tuning.csv` that all 4 grid points ran,
>   then report it as a finding (the paper's BPR also lost to PopRec on Beauty).



In [22]:
class BPR(torch.nn.Module):
    def __init__(
        self,
        n_users,
        n_items,
        d=50,
        l2=0.001,
        learning_rate=0.001,
        batch_size=128,
        init_std=0.01,
    ):
        super().__init__()
        if batch_size < 1:
            raise ValueError("batch_size must be at least 1")
        self.n_users = n_users
        self.n_items = n_items
        self.l2 = l2
        self.batch_size = batch_size
        self.user_emb = torch.nn.Embedding(n_users, d)
        self.item_emb = torch.nn.Embedding(n_items, d)
        torch.nn.init.normal_(self.user_emb.weight, mean=0.0, std=init_std)
        torch.nn.init.normal_(self.item_emb.weight, mean=0.0, std=init_std)
        self.optimizer = torch.optim.Adam(self.parameters(), lr=learning_rate)

    def train_one_epoch(self, train_seqs):
        seen = [set(sequence) for sequence in train_seqs]
        examples = [
            (user, item)
            for user, sequence in enumerate(train_seqs)
            for item in sequence
        ]
        random.shuffle(examples)
        self.train()
        total_loss = 0.0
        for start in range(0, len(examples), self.batch_size):
            batch = examples[start : start + self.batch_size]
            users = torch.tensor([user for user, _ in batch], dtype=torch.long)
            positives = torch.tensor([item for _, item in batch], dtype=torch.long)
            negatives = []
            for user, _ in batch:
                # Reject training positives
                item = random.randrange(self.n_items)
                while item in seen[user]:
                    item = random.randrange(self.n_items)
                negatives.append(item)
            negatives = torch.tensor(negatives, dtype=torch.long)
            user_vectors = self.user_emb(users)
            positive_vectors = self.item_emb(positives)
            negative_vectors = self.item_emb(negatives)
            difference = (user_vectors * (positive_vectors - negative_vectors)).sum(
                dim=1
            )

            ranking_loss = -torch.nn.functional.logsigmoid(difference).mean()
            penalty = (
                user_vectors.square().sum(dim=1)
                + positive_vectors.square().sum(dim=1)
                + negative_vectors.square().sum(dim=1)
            ).mean()
            loss = ranking_loss + self.l2 * penalty
            self.optimizer.zero_grad()
            loss.backward()
            self.optimizer.step()
            # Weight the final, possibly smaller batch correctly in the epoch avg.
            total_loss += loss.item() * len(batch)
        return total_loss / len(examples)

    @torch.no_grad()
    def score(self, users, histories):
        users = torch.tensor(list(users), dtype=torch.long)
        return (self.user_emb(users) @ self.item_emb.weight.T).cpu()

### 5.3 SASRec (Alina)

SASRec [1] looks at a user's items in order and guesses the next one. Every item gets a vector
and every position (1st, 2nd, ... 50th) gets one too, so the model knows the order. These go
through 2 self-attention blocks, and the output at the last position is the guess. An item's
score is the dot product of that guess with the item's vector.

**Self-attention.** At each position the model looks back at the earlier items, learns how
much each one matters for the next item, and takes a weighted mix of their vectors. So it can
figure out by itself whether the last item matters most or some older one does.

**Why causal.** We train every position at once: for `[a, b, c, d]` the input is `[a, b, c]`
and the targets are `[b, c, d]`. If a position could see the ones after it, it would just copy
the answer, so the causal mask blocks looking ahead. That's also how it works at test time,
when we only know the past.

**Settings.** Same as the paper (Sections III and IV-C): 2 blocks, 1 head, learned position
vectors, the same item vectors for input and output, the last `MAX_LEN = 50` items, dropout
0.5, Adam with learning rate 0.001, batch size 128. The loss is binary cross-entropy: at each
position the real next item should score high and one random item the user never had should
score low. Only `d` is tuned (Section 6). Item numbers start at 0, so we use `n_items` as the
padding number; its vector is always zero and `score` drops it.

**Compared to PopRec and BPR.** PopRec gives everyone the same popular items, cheap but not
personal. BPR is personal but ignores the order. SASRec uses the order and its list changes
with every new item, but it's the slowest to train, needs tuning, and moves a bit between
seeds. Our histories are short (about 8 items per user), so there isn't that much order to
learn.

In [23]:
# one block = attention + a small feed forward network,
# each part done like the paper: x + Dropout(part(LayerNorm(x)))
class SelfAttentionBlock(torch.nn.Module):
    def __init__(self, d, n_heads, dropout):
        super().__init__()
        self.norm1 = torch.nn.LayerNorm(d)
        self.attention = torch.nn.MultiheadAttention(
            d, n_heads, dropout=dropout, batch_first=True
        )
        self.norm2 = torch.nn.LayerNorm(d)
        self.feed_forward = torch.nn.Sequential(
            torch.nn.Linear(d, d),
            torch.nn.ReLU(),
            torch.nn.Dropout(dropout),
            torch.nn.Linear(d, d),
        )
        self.dropout = torch.nn.Dropout(dropout)

    def forward(self, x, causal_mask):
        h = self.norm1(x)
        attention_out, _ = self.attention(
            h, h, h, attn_mask=causal_mask, need_weights=False
        )
        x = x + self.dropout(attention_out)
        h = self.norm2(x)
        x = x + self.dropout(self.feed_forward(h))
        return x


class SASRec(torch.nn.Module):
    def __init__(
        self,
        n_users,
        n_items,
        d=50,
        n_blocks=2,
        n_heads=1,
        max_len=MAX_LEN,
        dropout=0.5,
        learning_rate=0.001,
        batch_size=128,
    ):
        super().__init__()
        self.n_items = n_items
        self.d = d
        self.max_len = max_len
        self.batch_size = batch_size

        # 0 is a real item, so n_items is the padding number
        self.pad = n_items
        self.item_emb = torch.nn.Embedding(n_items + 1, d, padding_idx=self.pad)
        self.pos_emb = torch.nn.Embedding(max_len, d)
        # small starting values like the paper's code (xavier init)
        torch.nn.init.xavier_normal_(self.item_emb.weight)
        torch.nn.init.xavier_normal_(self.pos_emb.weight)
        with torch.no_grad():
            self.item_emb.weight[self.pad] = 0  # padding vector stays 0

        self.dropout = torch.nn.Dropout(dropout)
        self.blocks = torch.nn.ModuleList()
        for _ in range(n_blocks):
            self.blocks.append(SelfAttentionBlock(d, n_heads, dropout))
        self.last_norm = torch.nn.LayerNorm(d)

        # True = can't look there, so position t only sees positions 0 to t
        self.causal_mask = torch.triu(
            torch.ones(max_len, max_len, dtype=torch.bool), diagonal=1
        )
        self.optimizer = torch.optim.Adam(self.parameters(), lr=learning_rate)

    # keep the last max_len items, fill the front with padding
    def pad_left(self, items):
        items = list(items)[-self.max_len :]
        padding = [self.pad] * (self.max_len - len(items))
        return padding + items

    # (batch, max_len) item numbers -> (batch, max_len, d), one guess per position
    def forward(self, seqs):
        x = self.item_emb(seqs) * (self.d**0.5)  # the paper's code scales by sqrt(d)
        x = x + self.pos_emb(torch.arange(self.max_len))
        x = self.dropout(x)
        not_pad = (seqs != self.pad).unsqueeze(-1).float()  # keeps padding at 0
        x = x * not_pad
        for block in self.blocks:
            x = block(x, self.causal_mask)
            x = x * not_pad
        return self.last_norm(x)

    # a random item the user never had in training
    def sample_negative(self, seen):
        item = random.randrange(self.n_items)
        while item in seen:
            item = random.randrange(self.n_items)
        return item

    def train_one_epoch(self, train_seqs):
        self.train()
        users = list(range(len(train_seqs)))
        random.shuffle(users)

        total_loss = 0.0
        n_batches = 0
        for start in range(0, len(users), self.batch_size):
            # for [a, b, c, d]: input [a, b, c], targets [b, c, d]
            inputs = []
            positives = []
            negatives = []
            for user in users[start : start + self.batch_size]:
                seq = train_seqs[user]
                if len(seq) < 2:
                    continue  # nothing to guess
                seen = set(seq)
                y_neg = []
                for _ in seq[1:]:
                    y_neg.append(self.sample_negative(seen))
                inputs.append(self.pad_left(seq[:-1]))
                positives.append(self.pad_left(seq[1:]))
                negatives.append(self.pad_left(y_neg))

            inputs = torch.tensor(inputs)
            positives = torch.tensor(positives)
            negatives = torch.tensor(negatives)

            outputs = self.forward(inputs)
            pos_scores = (outputs * self.item_emb(positives)).sum(dim=-1)
            neg_scores = (outputs * self.item_emb(negatives)).sum(dim=-1)

            # real next item -> label 1, random item -> label 0, padding skipped
            real = positives != self.pad
            bce = torch.nn.functional.binary_cross_entropy_with_logits
            pos_loss = bce(pos_scores[real], torch.ones_like(pos_scores[real]))
            neg_loss = bce(neg_scores[real], torch.zeros_like(neg_scores[real]))
            loss = pos_loss + neg_loss

            self.optimizer.zero_grad()
            loss.backward()
            self.optimizer.step()
            total_loss = total_loss + loss.item()
            n_batches = n_batches + 1
        return total_loss / n_batches

    @torch.no_grad()
    def score(self, users, histories):
        self.eval()
        inputs = []
        for history in histories:
            inputs.append(self.pad_left(history))
        last_outputs = self.forward(torch.tensor(inputs))[:, -1, :]
        # dot product with every real item (the padding row is left out)
        return last_outputs @ self.item_emb.weight[: self.n_items].T

## 6. Training and tuning (Alina)

Every trained model is tuned the same way, so the baselines get as much tuning as SASRec:

| Model | Tuned on validation | Grid |
|---|---|---|
| BPR | `d`, `l2` | `d` ∈ {10, 50}; `l2` ∈ {0.001, 0.01}: 4 points |
| SASRec | `d` | {10, 50}; everything else at the paper's values |
| PopRec | nothing | — |

The paper tries `d` ∈ {10, 20, 30, 40, 50} and `l2` ∈ {0.0001, …, 1}, and stops after 20
epochs without improvement (Section IV-B). We keep points inside those ranges but use fewer of
them, and stop after `PATIENCE = 10`, so that all training fits on our laptops (see the
limitations in Section 10).

1. Train every grid point with seed 1 and measure it on validation only.
2. Pick the point with the highest sampled validation NDCG@10. If two points tie to four
   decimals, take the smaller `d`, then the smaller `l2`.
3. Train the chosen point with seeds 1, 2 and 3 and measure it on test. Test is used only here,
   once.

Early stopping, for every trained model: after every epoch, measure sampled validation NDCG@10;
stop after `PATIENCE` epochs without improvement, or at `MAX_EPOCHS`; keep the model of the
best epoch. Full ranking is reported, never used to choose anything.

`train_model` sets the seed, builds the model with `ModelClass(n_users, n_items, **settings)`,
and after every epoch checks sampled NDCG@10 on validation. It keeps a copy of the best model
so far and stops after `PATIENCE` epochs without improvement (or at `MAX_EPOCHS`). It returns
the best model and its epoch, which is the `epochs` column in the CSV files.

`tune_and_test(method, ModelClass, grid)` does the 3 steps above and writes
`results/tuning.csv`, `results/test.csv` and `results/ranks.csv.gz`. Seed 1 of the chosen
settings is reused from step 1 instead of trained again, since the same seed gives the same
model. `show_results(method)` prints the tables from the files, so it also works with
`RUN_EXPERIMENTS = False`.

In [24]:
def train_model(ModelClass, settings, seed):  # noqa: N803 (it's a class)
    """Trains one model with early stopping, returns (best model, best epoch)."""
    set_seed(seed)
    model = ModelClass(n_users, n_items, **settings)

    best_ndcg = -1.0
    best_model = None
    best_epoch = 0
    bad_epochs = 0  # epochs in a row without a new best
    for epoch in range(1, MAX_EPOCHS + 1):
        start_time = time.time()
        loss = model.train_one_epoch(train_seqs)
        result = evaluate_sampled(
            model, train_seqs, valid_candidates
        )  # validation only
        ndcg = float(result["ndcg@10"])
        seconds = time.time() - start_time
        print(
            f"epoch {epoch}  loss {loss:.4f}  valid NDCG@10 {ndcg:.4f}  {seconds:.0f}s"
        )

        if ndcg > best_ndcg:
            best_ndcg = ndcg
            best_epoch = epoch
            best_model = copy.deepcopy(
                model
            )  # a real copy, training keeps changing model
            bad_epochs = 0
        else:
            bad_epochs = bad_epochs + 1
            if bad_epochs == PATIENCE:
                break

    print(f"best epoch: {best_epoch}   valid NDCG@10: {best_ndcg:.4f}")
    return best_model, best_epoch

In [25]:
# writes rows to a CSV in results/, replacing this method's old rows if there are any
def save_rows(path, rows, method):
    new_table = pd.DataFrame(rows)
    if os.path.exists(path):
        old_table = pd.read_csv(path)
        old_table = old_table[old_table["method"] != method]
        new_table = pd.concat([old_table, new_table])
    new_table.to_csv(path, index=False)


# highest NDCG@10 (4 decimals) first, then smaller d, then smaller l2
def tie_break_key(row):
    l2 = row["l2"]
    if l2 is None:
        l2 = 0  # SASRec has no l2
    return (-round(row["valid_ndcg10"], 4), row["d"], l2)


# tests one trained model with both protocols. gives back its rows for test.csv and,
# for seed 1, every user's rank for ranks.csv.gz (Section 8 uses them)
def test_one_model(method, model, seed, epochs):
    test_rows = []
    rank_rows = []
    sampled = evaluate_sampled(model, test_histories, test_candidates)
    full = evaluate_full(model, test_histories, test_answers)
    for protocol, result in [("sampled", sampled), ("full", full)]:
        test_rows.append(
            {
                "method": method,
                "seed": seed,
                "protocol": protocol,
                "hit10": float(result["hit@10"]),
                "ndcg10": float(result["ndcg@10"]),
                "epochs": epochs,
            }
        )
        if seed == 1:
            ranks = result["ranks"].tolist()
            for user in range(n_users):
                rank_rows.append(
                    {
                        "method": method,
                        "protocol": protocol,
                        "user": user,
                        "rank": ranks[user],
                    }
                )
    return test_rows, rank_rows


def tune_and_test(method, ModelClass, grid):  # noqa: N803 (it's a class)
    # step 1: every grid point with seed 1, on validation
    tuning_rows = []
    seed1_models = []
    for settings in grid:
        print(f"===== {method} {settings}  seed 1 =====")
        model, epochs = train_model(ModelClass, settings, seed=1)
        result = evaluate_sampled(model, train_seqs, valid_candidates)
        tuning_rows.append(
            {
                "method": method,
                "d": settings["d"],
                "l2": settings.get("l2"),  # empty in the CSV if there's no l2
                "valid_ndcg10": float(result["ndcg@10"]),
                "epochs": epochs,
            }
        )
        seed1_models.append(model)
    save_rows("results/tuning.csv", tuning_rows, method)

    # step 2: pick the best one
    best_row = sorted(tuning_rows, key=tie_break_key)[0]
    best_index = tuning_rows.index(best_row)
    best_settings = grid[best_index]
    print(f"chosen settings for {method}: {best_settings}")

    # step 3: seeds 1, 2, 3 on test (seed 1 is already trained from step 1)
    test_rows = []
    rank_rows = []
    for seed in TRAIN_SEEDS:
        if seed == 1:
            model = seed1_models[best_index]
            epochs = best_row["epochs"]
            seed1_model = model
        else:
            print(f"===== {method} {best_settings}  seed {seed} =====")
            model, epochs = train_model(ModelClass, best_settings, seed=seed)
        rows, ranks = test_one_model(method, model, seed, epochs)
        test_rows = test_rows + rows
        rank_rows = rank_rows + ranks

    save_rows("results/test.csv", test_rows, method)
    save_rows("results/ranks.csv.gz", rank_rows, method)
    return seed1_model


# prints the grid table and the test mean and std of one method from the CSV files
def show_results(method):
    if not os.path.exists("results/test.csv"):
        print("no results yet for", method)
        return
    tuning = pd.read_csv("results/tuning.csv")
    test = pd.read_csv("results/test.csv")
    test = test[test["method"] == method]
    if len(test) == 0:
        print("no results yet for", method)
        return
    tuning = tuning[tuning["method"] == method]
    if len(tuning) > 0:  # PopRec has no grid
        print(tuning.to_string(index=False))
        print()
    print(test.groupby("protocol")[["hit10", "ndcg10"]].agg(["mean", "std"]).round(4))

### 6.1 SASRec

Only `d` is tuned (10 or 50). `d = 10` won, so the demo (Section 9) loads
`results/sasrec_demo.pt` with `SASRec(n_users, n_items, d=10)`. One epoch takes about 5 s on a
laptop CPU and the whole run took about 35 minutes. Seeds 2 and 3 hit `MAX_EPOCHS = 100` while
still improving very slightly (about 0.002 over their last 10 epochs).

In [26]:
SASREC_GRID = [{"d": 10}, {"d": 50}]

if RUN_EXPERIMENTS:
    sasrec_model = tune_and_test("sasrec", SASRec, SASREC_GRID)
    torch.save(sasrec_model.state_dict(), "results/sasrec_demo.pt")  # for the demo

show_results("sasrec")

method  d  l2  valid_ndcg10  epochs
sasrec 10 NaN      0.367178      86
sasrec 50 NaN      0.347631      65

           hit10          ndcg10        
            mean     std    mean     std
protocol                                
full      0.0296  0.0009  0.0153  0.0003
sampled   0.5463  0.0014  0.3389  0.0017


In [27]:
BPR_GRID = [
    {"d": 10, "l2": 0.001},
    {"d": 10, "l2": 0.01},
    {"d": 50, "l2": 0.001},
    {"d": 50, "l2": 0.01},
]

if True:  # RUN_EXPERIMENTS:
    tune_and_test("bpr", BPR, BPR_GRID)

show_results("bpr")

===== bpr {'d': 10, 'l2': 0.001}  seed 1 =====
epoch 1  loss 0.6928  valid NDCG@10 0.0968  4s
epoch 2  loss 0.6733  valid NDCG@10 0.1857  5s
epoch 3  loss 0.6125  valid NDCG@10 0.2273  5s
epoch 4  loss 0.5360  valid NDCG@10 0.2508  5s
epoch 5  loss 0.4642  valid NDCG@10 0.2652  5s
epoch 6  loss 0.4040  valid NDCG@10 0.2743  5s
epoch 7  loss 0.3558  valid NDCG@10 0.2799  5s
epoch 8  loss 0.3176  valid NDCG@10 0.2832  5s
epoch 9  loss 0.2874  valid NDCG@10 0.2853  5s
epoch 10  loss 0.2631  valid NDCG@10 0.2866  5s
epoch 11  loss 0.2421  valid NDCG@10 0.2871  5s
epoch 12  loss 0.2260  valid NDCG@10 0.2873  5s
epoch 13  loss 0.2108  valid NDCG@10 0.2867  5s
epoch 14  loss 0.1994  valid NDCG@10 0.2864  5s
epoch 15  loss 0.1877  valid NDCG@10 0.2853  5s
epoch 16  loss 0.1778  valid NDCG@10 0.2844  5s
epoch 17  loss 0.1692  valid NDCG@10 0.2837  5s
epoch 18  loss 0.1620  valid NDCG@10 0.2826  5s
epoch 19  loss 0.1553  valid NDCG@10 0.2813  5s
epoch 20  loss 0.1496  valid NDCG@10 0.2807  5s
ep

### 6.2 PopRec

PopRec has nothing to tune and no randomness, so we build it once and write the same numbers
for all 3 seeds (`epochs` is 1, its one counting pass).

In [ ]:
if RUN_EXPERIMENTS:
    pop_model = PopRec(n_users, n_items)
    pop_model.train_one_epoch(train_seqs)
    pop_test_rows = []
    pop_rank_rows = []
    for seed in TRAIN_SEEDS:
        rows, ranks = test_one_model("poprec", pop_model, seed, epochs=1)
        pop_test_rows = pop_test_rows + rows
        pop_rank_rows = pop_rank_rows + ranks
    save_rows("results/test.csv", pop_test_rows, "poprec")
    save_rows("results/ranks.csv.gz", pop_rank_rows, "poprec")

show_results("poprec")

## 7. Results (Kyle)

Every training run writes its results to CSV files in results/. With
RUN_EXPERIMENTS = False, Sections 6 to 8 read these files instead of training.

| File | One row per | Columns |
|---|---|---|
| `results/tuning.csv` | grid point | `method, d, l2, valid_ndcg10, epochs` |
| `results/test.csv` | method, seed, protocol | `method, seed, protocol, hit10, ndcg10, epochs` |
| `results/ranks.csv.gz` | method, protocol, user (seed 1 only) | `method, protocol, user, rank` |

`protocol` is `sampled` or `full`, and `l2` is empty for models without it.

> **TODO** Delete this box when done.
> - [ ] **Matthew:** one figure for the main results table: Hit@10 and NDCG@10 per method
>   (PopRec, BPR, SASRec) with error bars (std over 3 seeds), sampled and full ranking side by
>   side. Read the numbers from `results/test.csv`. Full-ranking values are about 20 times
>   smaller than sampled ones, so use two panels instead of one shared axis.
> - [ ] **Kyle:** first check that every method was measured on the same users with the same
>   candidate lists: in `results/ranks.csv.gz` every method must have exactly one rank per user
>   (50,985) and protocol (the teacher asked for this check). Then the main table, like the
>   paper's Table III: one row per method (PopRec, BPR, SASRec), Hit@10 and NDCG@10 as
>   mean ± std over the 3 seeds, sampled and full ranking side by side, from `results/test.csv`.
> - [ ] **Kyle:** answer RQ2 in a short paragraph: does SASRec still rank above PopRec and BPR?
>   Compare our order with the paper's order from Section 1, not the exact numbers.

In [ ]:
# check every method has exactly one rank per user and protocol,
# so all methods were tested on the same users
ranks = pd.read_csv("results/ranks.csv.gz")
for (method, protocol), group in ranks.groupby(["method", "protocol"]):
    one_per_user = group["user"].is_unique and len(group) == n_users
    print(f"{method:7} {protocol:8} {len(group):6} ranks, one per user: {one_per_user}")
    assert one_per_user

In [ ]:
# mean +- std over the 3 seeds, sampled and full ranking side by side
METHODS = ["poprec", "bpr", "sasrec"]
test = pd.read_csv("results/test.csv")
summary = test.groupby(["method", "protocol"])[["hit10", "ndcg10"]].agg(["mean", "std"])

table = pd.DataFrame(index=[m for m in METHODS if m in summary.index.get_level_values(0)]) 
for protocol in ["sampled", "full"]:
    for metric, name in [("hit10", "Hit@10"), ("ndcg10", "NDCG@10")]:
        column = []
        for method in table.index:
            mean = summary.loc[(method, protocol), (metric, "mean")]
            std = summary.loc[(method, protocol), (metric, "std")]
            column.append(f"{mean:.4f} ± {std:.4f}")
        table[f"{name} ({protocol})"] = column
print(table.to_string())

## 8. Sampled evaluation vs full ranking: our main question (Kyle, Matthew)

> **TODO** Delete this box when done.
> - [ ] **Kyle:** RQ1: do the sampled protocol and full ranking put the methods in the same
>   order? Show both orders side by side (from the Section 7 table) and explain any change.
>   Krichene and Rendle [4] show that sampled metrics can change the order of methods; say
>   whether we see the same.
> - [ ] **Matthew:** results by the length of the test input history (`len(test_histories[u])`), as the
>   teacher suggested: groups 4–5, 6–10 and more than 10 items (5-core gives every user at
>   least 5 items, so at least 4 in the test history; check it). From `results/ranks.csv.gz`,
>   for PopRec, BPR and SASRec under both protocols: Hit@10 and NDCG@10 per group, plus the
>   number of users per group, in one figure.
> - [ ] **Matthew:** two or three sentences: does the gap between sampled and full ranking change with the
>   history length? Does SASRec gain more on longer histories?

## 9. Demo (Matthew)

> **TODO (Matthew)** The demo is due Sunday, so the video can be recorded on the final
> version. Delete this box when the cells work.
>
> **What the demo shows.** Given what one user reviewed so far, which products we recommend
> next. We compare SASRec (uses the order of the history) with PopRec (the same popular list
> for everyone), so the viewer sees that SASRec's list is personal and changes with the
> history while PopRec's never does.
>
> **The code to add here:**
> - [ ] Product names: the 5-core files only have `parent_asin`. Download
>   `meta_Industrial_and_Scientific.jsonl.gz` from the Amazon Reviews 2023 page (raw metadata
>   of the category). Keep only `parent_asin` and `title` for our 25,848 items (`item_ids`),
>   save that as `data/Industrial_and_Scientific/titles.csv.gz` and commit only that small
>   file, not the metadata. Then a cell that reads it into a dict `titles[item_number]`.
> - [ ] Load the models once:
>   ```python
>   demo_sasrec = SASRec(n_users, n_items, d=10)  # d = 10 won the tuning (Section 6.1)
>   demo_sasrec.load_state_dict(torch.load("results/sasrec_demo.pt"))
>   demo_pop = PopRec(n_users, n_items)
>   demo_pop.train_one_epoch(train_seqs)
>   ```
> - [ ] `show_user(u)`: takes `history = test_histories[u]` and the real next item
>   `answer = test_answers[u]`, then prints
>   1. the history as titles, oldest first, numbered;
>   2. SASRec's top 10 from `recommend(demo_sasrec, [u], [history], 10)[0]`, as titles;
>   3. PopRec's top 10 the same way;
>   4. a mark like `<-- real next item` next to `answer` wherever it appears in a list.
>   Shorten long titles (for example to 70 characters) so each list fits on one screen.
> - [ ] Pick the users, and say in the video that we picked them:
>   - users A and B: SASRec has the real next item in its top 10 (`rank <= 10` for
>     `method == "sasrec"`, `protocol == "full"` in `results/ranks.csv.gz`), with clearly
>     different kinds of products (for example lab supplies vs 3D-printing parts) and a
>     history of 5 to 15 items so it fits on screen;
>   - user C: SASRec misses (rank > 10).
>   Write the three user numbers in the cell so the demo always shows the same users.
>
> **The video** (about 3 minutes; the whole talk, video included, must fit in 20 minutes):
> 1. One sentence on screen: "Given what a user reviewed so far, we recommend the next
>    product."
> 2. User A: the history, then SASRec's top 10 with the real next item marked. Pause and say
>    why the list makes sense (it follows the most recent items).
> 3. User A with PopRec: the generic popular list, the real item is not in it. This is the
>    baseline.
> 4. User B with different taste: SASRec's list changes completely, PopRec's is the same list
>    as for user A.
> 5. User C: SASRec misses. An honest example of its limits (for example a very short history
>    or a next item unlike anything before it).
>
> - [ ] Record only the outputs (run the cells and scroll; no code walk-through, the teacher
>   says that is not a demo), 1080p (macOS: QuickTime, File → New Screen Recording), a short
>   title line above each scene. No voice-over needed, we comment live when pausing in class.
> - [ ] Save it as `demo.mp4` and upload it to Moodle with the hand-in. Do not commit it.
> - [ ] Agree with the team who comments at which pause (all four of us must speak).

## 10. Discussion (Matthew)

> **TODO (Matthew)** Delete this box when done.
> - [ ] **Observations**, once Sections 7 and 8 have their results: write them in your own
>   words under "Observations" below. Things to look at: do sampled and full ranking put the
>   methods in the same order (RQ1)? Does SASRec still beat PopRec and BPR (RQ2)? How much
>   smaller are the full-ranking numbers (about 20 times for SASRec)? Does the gap change with
>   history length (Section 8)? `d = 10` beat `d = 50` for SASRec, maybe because our data is
>   sparse.
> - [ ] Read the limitations and future work below and add anything from your own parts (for
>   example what the repeated user–item pairs and timestamp ties in Section 3.2 mean for us).
> - [ ] **Conclusion:** a short paragraph at the end: the answer to RQ1 first (it is our main
>   question), then RQ2; which conclusions agree with the paper, which don't, and possible
>   reasons. Everyone reviews it.
> - [ ] **Monday, the final run** for the hand-in: make sure every TODO box is gone,
>   `RUN_EXPERIMENTS = False`, Restart Kernel and Run All, save the notebook **with outputs**
>   and upload that file to Moodle. Don't commit it (pre-commit strips the outputs).

### Observations

### Limitations

- **Smaller tuning grid than the paper.** The paper tries `d` in {10, 20, 30, 40, 50} and `l2`
  from 0.0001 to 1. We only tried `d` in {10, 50} (and `l2` in {0.001, 0.01} for BPR) so that
  everything fits on our laptops. A setting in between might have done better.
- **We stop earlier.** We stop after `PATIENCE = 10` epochs without improvement, the paper
  waits 20. And two of the three SASRec seeds (2 and 3) were still improving a tiny bit when
  they hit `MAX_EPOCHS = 100`, by about 0.002 NDCG@10 over their last 10 epochs, so their
  numbers are probably a little lower than they could be.
- **Only one category.** We only used `Industrial_and_Scientific`, which is also a different
  kind of product than the paper's Beauty and Games. That's why we compare the order of the
  methods and not the exact numbers, but it also means we can't say our results hold for
  other categories.

### Future work

- **A time-based split.** Amazon Reviews 2023 also comes with an official split by date
  (`timestamp_w_his`): everything before a cutoff date is for training and everything after
  is for testing. That's closer to how a recommender is really used than leave-one-out, where
  one user's test item can be older than another user's training items.
- **More categories.** Now that the whole pipeline works, running it on more categories
  (for example the bigger `Video_Games`) would show whether the order of the methods, and the
  sampled vs full ranking question, hold beyond one dataset.

### Conclusion

## 11. References

[1] W.-C. Kang and J. McAuley, "Self-attentive sequential recommendation," in *Proc. IEEE Int.
Conf. Data Mining (ICDM)*, 2018, pp. 197–206. arXiv:1808.09781.

[2] Y. Hou, J. Li, Z. He, A. Yan, X. Chen, and J. McAuley, "Bridging language and items for
retrieval and recommendation," arXiv:2403.03952v1, 2024. [Paper](https://arxiv.org/abs/2403.03952v1).
Dataset: [Amazon Reviews 2023](https://amazon-reviews-2023.github.io/).

[3] S. Rendle, C. Freudenthaler, Z. Gantner, and L. Schmidt-Thieme, "BPR: Bayesian
personalized ranking from implicit feedback," in *Proc. Conf. Uncertainty in Artificial
Intelligence (UAI)*, 2009, pp. 452–461.

[4] W. Krichene and S. Rendle, "On sampled metrics for item recommendation," in *Proc. ACM
SIGKDD Int. Conf. Knowledge Discovery and Data Mining (KDD)*, 2020, pp. 1748–1757.

[5] M. Ferrari Dacrema, P. Cremonesi, and D. Jannach, "Are we really making much progress? A
worrying analysis of recent neural recommendation approaches," in *Proc. ACM Conf.
Recommender Systems (RecSys)*, 2019, pp. 101–109. arXiv:1907.06902.

[6] A. Paszke et al., "PyTorch: An imperative style, high-performance deep learning library,"
in *Advances in Neural Information Processing Systems (NeurIPS)*, vol. 32, 2019.

[7] W. McKinney, "Data structures for statistical computing in Python," in *Proc. 9th Python
in Science Conf. (SciPy)*, 2010, pp. 56–61.

[8] J. D. Hunter, "Matplotlib: A 2D graphics environment," *Computing in Science &
Engineering*, vol. 9, no. 3, pp. 90–95, 2007.